# 60 — Prospective Protocol Freeze

Congela **antes de observar resultados futuros** el protocolo de M7: fechas, target, métricas, thresholds, modelo/calibrador, scoring contract y reglas de gobernanza.

Este notebook es el preregistro técnico de la evaluación prospectiva. TEST 2025 y OOT 2026 permanecen consumidos.


In [1]:
from argos.common.paths import ProjectPaths
from argos.config.prospective import load_prospective_spec
from argos.config.production import load_production_spec

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()
M7_SPEC = load_prospective_spec(paths.configs / "prospective.toml")
M6_SPEC = load_production_spec(paths.configs / "production.toml")
(paths.reports / "prospective").mkdir(parents=True, exist_ok=True)
(paths.root / "artifacts" / "prospective").mkdir(parents=True, exist_ok=True)

m6_gate = paths.docs / "decisions" / "M6_GATE.md"
assert m6_gate.exists() and "M6 ✅ CLOSED" in m6_gate.read_text(encoding="utf-8"), "M6 must be CLOSED before M7."
print("M7 schema:", M7_SPEC.schema_version)
print("Protocol freeze:", M7_SPEC.core.protocol_freeze_date)
print("Prospective start:", M7_SPEC.core.prospective_start)


M7 schema: m7-prospective-v2
Protocol freeze: 2026-09-23
Prospective start: 2026-09-24


In [2]:
from argos.prospective.protocol import PROTOCOL_IMPLEMENTATION, freeze_m7_protocol

print("implementation:", PROTOCOL_IMPLEMENTATION)
result = freeze_m7_protocol(
    root=paths.root,
    prospective_spec=M7_SPEC,
    production_spec=M6_SPEC,
    reports_root=paths.reports,
    docs_root=paths.docs,
)
print("protocol status:", result.status)
result.validation


implementation: m7-60-protocol-freeze-v2
protocol status: created


,check_id,category,blocking,passed,observed,requirement
0,m6_closed_prerequisite,prerequisite,True,True,"{""path"": ""/Users/macm1pro/Desktop/projects/arg...",M7 starts only after formal M6 closure
1,prospective_start_after_freeze,temporal_governance,True,True,"{""freeze"": ""2026-09-23"", ""start"": ""2026-09-24""}",all M7 evaluation cohorts occur strictly after...
2,consumed_holdouts_locked,evaluation_governance,True,True,"{""oot_year"": 2026, ""test_year"": 2025}",TEST 2025 and OOT 2026 stay consumed and canno...
3,human_review_preserved,governance,True,True,"{""m6"": true, ""m7"": true}",M7 remains human-in-the-loop
4,automatic_promotion_disabled,governance,True,True,"{""m6_production_promotion"": false, ""m7_auto_pr...",prospective evidence cannot trigger automatic ...
5,protocol_inputs_present,lineage,True,True,"{""missing"": []}",all protocol inputs and frozen model assets ar...
6,protocol_manifest_immutable,lineage,True,True,"{""path"": ""/Users/macm1pro/Desktop/projects/arg...",protocol freeze is persisted once and subseque...
7,protocol_manifest_verified,lineage,True,True,"{""core_mismatches"": {}, ""hash_mismatches"": {},...",frozen M7 hashes still match the exact V1 asse...


In [3]:
blocking = result.validation.loc[result.validation["blocking"]]
print("60 ✅ PASS" if result.ready_for_prospective_collection else "60 ❌ FAIL")
print(f"Blocking checks: {int(blocking['passed'].sum())}/{len(blocking)}")
print("Protocol manifest:", result.manifest_path)
assert result.ready_for_prospective_collection


60 ✅ PASS
Blocking checks: 8/8
Protocol manifest: /Users/macm1pro/Desktop/projects/argos/argos-code/artifacts/manifests/m7_protocol_freeze.json
